#Перевірка статиcтичних гіпотез. Z-test.



0. Зчитайте дані з `data.csv` у змінну data, яка має тип pandas.DataFrame. Ми будемо далі працювати з цією змінною.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/1. IT study/Data Loves Academy/Data Analyst/15. Модуль 15/homeworks/data.csv')

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats   #бібліотека для наукових обчислень із пакетом stats

In [4]:
data.head()

,Unnamed: 0,data
0,0,17.499453
1,1,19.662399
2,2,7.182823
3,3,29.841625
4,4,9.239386


Запустіть код нижче. Для коректної робити всіх подальших методів дані мають бути у вигляді одновимірного `numpy.ndarray` вектора та мати розмірність (100,). Така розмірність означає, що дані - одновимірні (колонка), якщо б розмірність була (100, 1), то дані сприймаються методами як двовимірні (таблиця), хоча для нас виглядати можуть так само.

In [5]:
data = data.data.values
data.shape

(100,)

**Завдання 1**. (7 балів)
Зчитані дані - це сума покупок в доларах наших покупців на сайті протягом липня 2025 року.

До цього місяця, ми мали продажі в середньому на чек $20.

Необхідно зʼясувати, чи вийшло у нас статистично значущо **підвищити** середній чек за липень 2025?

Аби дати відповідь - ми проведемо z-test на рівні значущості $\alpha=0.05$ двома способами. В цьому завданні ми будемо виконувати обчислення "вручну" з використанням бібліотек numpy та scipy.stats подібно до прикладу в лекції. Для цього виконайте наступні 7 кроків. Правильне виконання кожного з кроків оцінюється в 1 бал.

1. Запишіть параметри задачі у змінні Python:
    - `sample_mean` - $\bar{x}$, середнє значення в вибірці
    - `population_mean` - $\mu_0$, середнє значення в популяції (тобто те, з яким ми порівнюємо середнє вибіркове значення)
    - `population_std` - $\sigma$, вибіркове стандартне відхилення, яке ми вважаємо, що є рівним ст. відх. популяції, адже маємо достатньо велику вибірку.
    - `sample_size` - $n$, розмір вибірки
    - `alpha` - $\alpha$ рівень значущості
    
    **Увага!** Для обчислення стандартного відхилення маємо скористатись функцією `np.std(your_dataframe, ddof=1)`. Чому так - розбираємо в лекції про t-test.


2. Визначте для цієї задачі:
    - якою є гіпотеза $H_0$
    - якою є альтернативна гіпотеза $H_a$
    - з яким типом тесту ми маємо справу - лівосторонній, правосторонній чи двосторонній.

3. Обчисліть стандартну помилку SE.
4. Розрахуйте z-статистику.
5. Знайдіть критичне z-значення з допомогою бібліотеки stats.
6. Обчисліть p-value з допомогою бібліотеки stats.
7. Прийміть рішення, чи відхиляєте ви гіпотезу $H_0$. Для прийняття рішення зробіть і порівняння z-статистики з критичним значенням, і проаналізуйте p-value.

### Крок 1: Визначаємо параметри задачі

In [6]:
#1.1 - Задаємо всі відомі параметри


sample_mean = data.mean()                 # x̄  - середній чек у вибірці у липні 2025 (щось ця риска вгорі з'їжджає праворуч від х)
population_mean = 20                      # μ₀ - попередній середній чек, $ (те, з чим порівнюємо)
population_std = np.std(data, ddof=1)     # σ - стандартне відхилення, $
sample_size = len(data)                   # n - кількість чеків у вибірці
alpha = 0.05                              # α - рівень значущості (ймовірність помилки I роду)


print(f"Середній чек у вибірці у липні 2025: ${sample_mean:.4f}")
print(f"Попередній середній чек: ${population_mean}")
print(f"Стандартне відхилення: ${population_std:.4f}")
print(f"Розмір вибірки (кількість чеків у вибірці): {sample_size}")
print(f"Рівень значущості: {alpha}")

Середній чек у вибірці у липні 2025: $19.3776
Попередній середній чек: $20
Стандартне відхилення: $6.2536
Розмір вибірки (кількість чеків у вибірці): 100
Рівень значущості: 0.05


### Крок 2: Формулюємо гіпотези


**Гіпотези**   

$H_0: μ = 20$ (середній чек у липні 2025 не мінився і залишився, як раніше)  
$H_a: μ > 20$ (середній чек статистично значущо зріс і перевищує $20)  
Тип тесту: правосторонній (right-tailed)


**Чому саме правосторонній тест?**

Ми з'ясовуємо чи вдалося нам у липні 2025  **підвищити** чек (а не чи впав чек, чи просто відрізняється).

### Крок 3: Обчислюємо стандартну помилку

Формула:

$$
SE = \frac{\sigma}{\sqrt{n}}
$$

Тобто середня різниця між вибірковим середнім $\bar{x}$ і справжнім середнім $\mu$.

In [7]:
#1.3 - Обчислюємо стандартну помилку

# Формула: SE = σ / √n
standard_error = population_std / np.sqrt(sample_size)  #np. - використвуємо пакет з бібліотеки numpy

print("=" * 60)
print("ОБЧИСЛЕННЯ СТАНДАРТНОЇ ПОМИЛКИ:")
print("=" * 60)
print("Формула: SE = σ / √n")
print(f"SE = {population_std:.4f} / √{sample_size}")
print(f"SE = {population_std:.4f} / {np.sqrt(sample_size)}")
print(f"SE = {standard_error:.4f}")
print()
print("Інтерпретація: В середньому, вибіркове середнє відхиляється")
print(f"від справжнього середнього на ±{standard_error:.4f}")



ОБЧИСЛЕННЯ СТАНДАРТНОЇ ПОМИЛКИ:
Формула: SE = σ / √n
SE = 6.2536 / √100
SE = 6.2536 / 10.0
SE = 0.6254

Інтерпретація: В середньому, вибіркове середнє відхиляється
від справжнього середнього на ±0.6254


### Крок 4: Розраховуємо Z-статистику

$$
Z = \frac{\bar x - \mu_0}{SE}
$$

In [8]:
#1.4 - Розраховуємо Z-статистику

# Формула: z = (x̄ - μ₀) / SE
z_statistic = (sample_mean - population_mean) / standard_error

print("=" * 60)
print("РОЗРАХУНОК Z-СТАТИСТИКИ:")
print("=" * 60)
print("Формула: Z = (x̄ - μ₀) / SE")
print(f"Z = ({sample_mean:.4f} - {population_mean}) / {standard_error:.4f}")
print(f"Z = {(sample_mean - population_mean):.4f} / {standard_error:.4f}")
print(f"Z = {z_statistic:.4f}")
print()
print("Інтерпретація:")
print(f"Наше вибіркове середнє відхиляється на {z_statistic:.4f} стандартних")
print("помилок від гіпотетичного середнього.")
print()
if z_statistic > 2:
    print("Це досить велике відхилення. (більше 2 стандартних помилок)")
elif z_statistic > 1:
    print("Це помірне відхилення (між 1 та 2 стандартними помилками)")
else:
    print("Це невелике відхилення (менше 1 стандартної помилки)")

РОЗРАХУНОК Z-СТАТИСТИКИ:
Формула: Z = (x̄ - μ₀) / SE
Z = (19.3776 - 20) / 0.6254
Z = -0.6224 / 0.6254
Z = -0.9953

Інтерпретація:
Наше вибіркове середнє відхиляється на -0.9953 стандартних
помилок від гіпотетичного середнього.

Це невелике відхилення (менше 1 стандартної помилки)


### Крок 5: Знаходимо критичне значення

- Для правостороннього тесту з $α=0.05$ нам потрібна точка, що відсікає $95\%$ зліва
- Тому передаємо `1 - alpha = 0.95`


In [9]:
#1.5 - Знаходимо критичне значення

# Для правостороннього тесту з α = 0.05
z_critical = stats.norm.ppf(1 - alpha)

print("=" * 60)
print("КРИТИЧНЕ ЗНАЧЕННЯ:")
print("=" * 60)
print(f"Для рівня значущості α = {alpha} (правосторонній тест)")
print(f"Критичне z-значення = {z_critical:.4f}")
print()
print("Що це означає?")
print(f"Якщо Z-статистика > {z_critical:.4f}, то результат настільки")
print("незвичайний, що ми не віримо у випадковість за гіпотезою Н0.")
print(f"Ймовірність отримати Z > {z_critical:.4f} випадково = {alpha*100}%")

КРИТИЧНЕ ЗНАЧЕННЯ:
Для рівня значущості α = 0.05 (правосторонній тест)
Критичне z-значення = 1.6449

Що це означає?
Якщо Z-статистика > 1.6449, то результат настільки
незвичайний, що ми не віримо у випадковість за гіпотезою Н0.
Ймовірність отримати Z > 1.6449 випадково = 5.0%


### Крок 6: Обчислюємо p-value

Для обчислення p-value в Python використовуємо функцію `stats.norm.cdf()`.

- `cdf` (Cumulative Distribution Function) обчислює площу під кривою ЗЛІВА від точки
- `stats.norm.cdf(z)` дає ймовірність P(Z ≤ z)
- Але для правостороннього тесту нам потрібна площа СПРАВА: P(Z ≥ z)
- Тому віднімаємо від 1: `1 - stats.norm.cdf(z)`


In [10]:
#1.6 - Обчислюємо p-value

# Для правостороннього тесту: p-value = P(Z > z_statistic)
p_value = 1 - stats.norm.cdf(z_statistic)

print("=" * 60)
print("P-VALUE:")
print("=" * 60)
print(f"p-value = {p_value:.6f}")
print()
print("Що це означає?")
print(f"Якщо H₀ правильна (середній чек не змінився), то ймовірність")
print(f"випадково отримати середній чек ≥ {sample_mean:.4f} становить {p_value:.6f}")
print(f"або приблизно {p_value*100:.4f}%")
print()
if p_value < 0.001:
    print("Це НАДЗВИЧАЙНО малоймовірно! (менше 0.1%)")
elif p_value < 0.01:
    print("Це дуже малоймовірно! (менше 1%)")
elif p_value < 0.05:
    print("Це малоймовірно! (менше 5%)")
else:
    print("Це цілком можливо випадково")

P-VALUE:
p-value = 0.840216

Що це означає?
Якщо H₀ правильна (середній чек не змінився), то ймовірність
випадково отримати середній чек ≥ 19.3776 становить 0.840216
або приблизно 84.0216%

Це цілком можливо випадково


### Крок 7: Приймаємо рішення

#### Порівняння Z-статистики з критичним значенням

In [11]:
#1.7 - Приймаємо рішення: порівнюємо Z-статистику з критичним значенням

print("=" * 60)
print("Порівняння Z-статистики з критичним значенням")
print("=" * 60)
print(f"Z-статистика = {z_statistic:.4f}")
print(f"Критичне значення = {z_critical:.4f}")
print(f"Порівняння: {z_statistic:.4f} {'>' if z_statistic > z_critical else '≤'} {z_critical:.4f}")

if z_statistic > z_critical:
    print("Висновок: Z-статистика ПЕРЕВИЩУЄ критичне значення")
    decision1 = "ВІДХИЛЯЄМО H₀"
else:
    print("Висновок: Z-статистика НЕ перевищує критичне значення")
    decision1 = "НЕ ВІДХИЛЯЄМО H₀"

print(f"Рішення: {decision1}")

Порівняння Z-статистики з критичним значенням
Z-статистика = -0.9953
Критичне значення = 1.6449
Порівняння: -0.9953 ≤ 1.6449
Висновок: Z-статистика НЕ перевищує критичне значення
Рішення: НЕ ВІДХИЛЯЄМО H₀


**Порівняння p-value з рівнем значущості**

- p-value $≤ 0.05$ — відхиляємо нульову гіпотезу $H_0$
- p-value $> 0.05$ — не відхиляємо нульову гіпотезу $H_0$

Рівень значущості $\alpha = 0.05$ — це заздалегідь визначена межа. За стандартним правилом, якщо $p \leq \alpha$, результат є статистично значущим.

In [15]:
#1.7 - Приймаємо рішення: порівнюємо p-value з рівнем значущості

print("\n" + "=" * 60)
print("МЕТОД 2: Порівняння p-value з рівнем значущості")
print("=" * 60)
print()
print(f"p-value = {p_value:.6f}")
print(f"Рівень значущості α = {alpha}")
print(f"Порівняння: {p_value:.6f} {'≤' if p_value < alpha else '>'} {alpha}")

if p_value < alpha:
    print("Висновок: p-value МЕНШЕ за рівень значущості")
    decision2 = "ВІДХИЛЯЄМО H₀"
else:
    print("Висновок: p-value НЕ менше за рівень значущості")
    decision2 = "НЕ ВІДХИЛЯЄМО H₀"

print(f"Рішення: {decision2}")


МЕТОД 2: Порівняння p-value з рівнем значущості

p-value = 0.840216
Рівень значущості α = 0.05
Порівняння: 0.840216 > 0.05
Висновок: p-value НЕ менше за рівень значущості
Рішення: НЕ ВІДХИЛЯЄМО H₀


###**Завдання 2.** (3 бали)
Виконайте обчислення z-test з використанням бібліотеки statsmodels.

Отримайте z-статистику та р-значення.

Виведіть p-значення та зробіть висновок, чи ми маємо достатньо доказів, аби стверджувати, що середній чек зріс.

Чи зійшлись значення z-статистики та р-значення в цьому завданні з попередніми обчисленнями?

In [13]:
from statsmodels.stats import weightstats as stests

In [14]:
print("\n" + "=" * 60)
print("ВИКОНАННЯ Z-ТЕСТУ:")
print("=" * 60)

# Виконуємо правосторонній z-тест
z_stat_sm, p_val_sm = stests.ztest(data,                   # наші дані
                                   value=population_mean,  # μ₀ = 20
                                   alternative='larger')   # H₁: μ > 20

print(f"Результати z-тесту:")
print(f"  Z-статистика: {z_stat_sm:.4f}")
print(f"  P-value: {p_val_sm:.6f}")
print()

# Інтерпретація результатів
print("ІНТЕРПРЕТАЦІЯ:")
print("-" * 40)
if p_val_sm < alpha:
    print(f"✅ ВІДХИЛЯЄМО H₀")
    print(f"   p-value ({p_val_sm:.6f}) < α ({alpha})")
    print("   Висновок: Середній чек у липні 2025 зріс")
else:
    print(f"❌ НЕ ВІДХИЛЯЄМО H₀")
    print(f"   p-value ({p_val_sm:.6f}) ≥ α ({alpha})")
    print("   Висновок: Немає достатніх доказів про вищий середній чек у липні 2025, аніж до липня")

# Порівняння з ручними розрахунками
print("\n" + "=" * 60)
print("ПОРІВНЯННЯ З ПОПЕРЕДНІМИ ОБЧИСЛЕННЯМИ:")
print("=" * 60)
print(f"Z-статистика:")
print(f"  Ручний розрахунок: {z_statistic:.4f}")
print(f"  Statsmodels:       {z_stat_sm:.4f}")
print(f"  Різниця:           {abs(z_statistic - z_stat_sm):.4f}")


ВИКОНАННЯ Z-ТЕСТУ:
Результати z-тесту:
  Z-статистика: -0.9953
  P-value: 0.840216

ІНТЕРПРЕТАЦІЯ:
----------------------------------------
❌ НЕ ВІДХИЛЯЄМО H₀
   p-value (0.840216) ≥ α (0.05)
   Висновок: Немає достатніх доказів про вищий середній чек у липні 2025, аніж до липня

ПОРІВНЯННЯ З ПОПЕРЕДНІМИ ОБЧИСЛЕННЯМИ:
Z-статистика:
  Ручний розрахунок: -0.9953
  Statsmodels:       -0.9953
  Різниця:           0.0000
